### Data Set Selection

Dataset: Weather Prediction\
Link: https://www.kaggle.com/datasets/ananthr1/weather-prediction?resource=download\
Target: Multivariate Time-series Classification


### Why Sequence Model?

This data set requires a sequence model because tomorrow’s weather depends not just on today’s conditions but on patterns over several days like gradual cooling, accumulated wind, and shifting precipitation. The prediction problem is sequence-based. An LSTM (or GRU) can capture these temporal dependencies by processing the last 'n' days as a single sequence and learning how those patterns influence tomorrow’s weather category.

### Part 1

In [26]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, make_scorer, precision_score, recall_score, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv("seattle-weather.csv")

data.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [27]:
print(data['weather'].unique())

['drizzle' 'rain' 'sun' 'snow' 'fog']


Changing the weather values into integer values
- 'drizzle' -> 0
- 'rain' -> 1
- 'sun' -> 2
- 'snow' -> 3
- 'fog' -> 4

In [21]:
data['weather'] = data['weather'].replace({'drizzle': 0, 'rain': 1, 'sun': 2, 'snow': 3, 'fog': 4})
data.head()

C:\Users\Neil\AppData\Local\Temp\ipykernel_27584\1436077711.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['weather'] = data['weather'].replace({'drizzle': 0, 'rain': 1, 'sun': 2, 'snow': 3, 'fog': 4})


,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,0
1,2012-01-02,10.9,10.6,2.8,4.5,1
2,2012-01-03,0.8,11.7,7.2,2.3,1
3,2012-01-04,20.3,12.2,5.6,4.7,1
4,2012-01-05,1.3,8.9,2.8,6.1,1


In [22]:
numeric_cols = data.drop(columns=['weather', 'date']).columns
cat_data = data[['weather', 'date']] # Extract the categorical column separately

scalar = StandardScaler()
scaled_data = scalar.fit_transform(data[numeric_cols]) # Scale the numeric features

df = pd.DataFrame(scaled_data, columns=numeric_cols) # Create DataFrame with scaled features
df[['weather', 'date']] = cat_data #Add the categorical column back

df.head()

,precipitation,temp_max,temp_min,wind,weather,date
0,-0.453650,-0.495299,-0.644212,1.014980,0,2012-01-01
1,1.178598,-0.794731,-1.082347,0.875833,1,2012-01-02
2,-0.333852,-0.645015,-0.206077,-0.654780,1,2012-01-03
3,2.586224,-0.576962,-0.524720,1.014980,1,2012-01-04
4,-0.258978,-1.026111,-1.082347,1.989006,1,2012-01-05


In [23]:
numeric_only = df.drop(columns=['weather', 'date'])
vif = pd.DataFrame()
vif["features"] = numeric_only.columns # Assign feature names
vif["VIF Factor"] = [variance_inflation_factor(numeric_only.values, i) for i in range(numeric_only.shape[1])] # Compute VIF for each feature
vif.round(2)
print(vif)

        features  VIF Factor
0  precipitation    1.232497
1       temp_max    4.916071
2       temp_min    4.652557
3           wind    1.137168


In [ ]:
def create_sequences(features_df, weather_labels, window_size):
    X = []
    y = []
    for i in range(len(features_df) - window_size):
        X.append(features_df.iloc[i:i+window_size].values)
        y.append(weather_labels.iloc[i + window_size])
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(numeric_only, df['weather'], 7)

print(X_seq.shape)
print(y_seq.shape)
X_train, X_temp, y_train, y_temp = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

(1454, 7, 4)
(1454,)


In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)
val_loader = DataLoader(val_dataset, batch_size=32)

### RNN (PyTorch)

The RNN I am using is built from PyTorch. From the input to the hidden layer, pytorch uses a linear transformation. From the hidden state out, it uses tanh as the activation function.

In [ ]:
class WeatherRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        #Uses tanh here
        out, h_n = self.rnn(x)
        out = h_n[-1]
        #No activation here
        out = self.fc(out)
        return out
    
    def fit(self, epochs):
        

In [32]:
input_size = X_train.shape[2]
model = WeatherRNN(input_size=input_size, hidden_size=64, num_classes=5)

